# PathSnap: Distributed Map-Matching & Route Reconstruction
### UE24CS352A - Machine Learning | PES University, RR Campus
---
This interactive walkthrough demonstrates:
1. **Metric Projection & Road Graph Setup**
2. **Candidate Extraction & Kinematic Features**
3. **Tuned LightGBM Inference & HMM Viterbi Route Reconstruction**
4. **Benchmarking & Accuracy Verification**
5. **Interactive Folium Geographical Visualizations**

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import joblib
from data_processing.loader import load_trip
from features.candidate_extractor import CandidateExtractor
from hmm.viterbi import HMMMapMatcher
from visualizations.folium_map import generate_trip_folium_map
print('All modules loaded successfully!')

## 1. Load Trip Data (.track, .nodes, .arcs, .route)

In [2]:
trip = load_trip('../data/student_test', '00000007', load_route=True)
print(f'Trip ID: {trip.trip_id}')
print(f'GPS Points:        {len(trip.track)}')
print(f'Road Network Nodes:{len(trip.nodes)}')
print(f'Road Network Arcs: {len(trip.arcs)}')
print(f'Ground Truth Arcs: {len(trip.route)}')
print(f'Center Lat/Lon:    {trip.center_lat:.4f}')

## 2. Spatial Candidate Generation & Feature Extraction

In [3]:
extractor = CandidateExtractor(trip, top_k=8)
cands = extractor.extract_candidates_for_point(0)
pd.DataFrame(cands)[['candidate_rank', 'arc_id', 'dist_to_segment', 'heading_diff', 'gps_speed', 'is_match']]

## 3. Tuned LightGBM + HMM Viterbi Sequential Decoding

In [4]:
model = joblib.load('../saved_models/lightgbm_tuned.joblib')
matcher = HMMMapMatcher(trip, model)
res = matcher.run_reconstruction(extractor)
print(f'Pointwise Accuracy:   Clf={res["observation_acc_clf"]:.2f}%  -->  HMM={res["observation_acc_hmm"]:.2f}%')
print(f'Route Continuity:     Clf={res["continuity_clf"]:.2f}%  -->  HMM={res["continuity_hmm"]:.2f}%')
print(f'Reconstructed Arcs:   {len(res["reconstructed_route"])} arcs')
print(f'Reference Arcs:       {len(trip.route)} arcs')

## 4. Multi-Model Benchmark Summary (Section 3.1)

In [5]:
pd.read_csv('../results/model_comparison.csv')

## 5. Bayesian Hyperparameter Tuning Impact (Section 4)

In [6]:
pd.read_csv('../results/tuned_vs_untuned.csv')